In [ ]:

!pip install -q ultralytics einops timm imageio gdown

import os, sys
if not os.path.exists('/kaggle/working/MotionBERT'):
    !git clone https://github.com/Walter0807/MotionBERT.git /kaggle/working/MotionBERT
sys.path.insert(0, '/kaggle/working/MotionBERT')

import torch
import numpy as np
import cv2

# Setup device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch : {torch.__version__}')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')

# Create checkpoint directory
CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# MotionBERT standard temporal window
WINDOW = 243
print(f'WINDOW = {WINDOW}')
print('✅ Setup complete!')


PyTorch : 2.9.0+cu126
Device : cuda
GPU : Tesla T4
WINDOW = 243
✅ Setup complete!


In [9]:
import os
import glob

# ============================================================
# DATASET PATHS
# ============================================================
AP_ROOT = '/kaggle/input/datasets/haruxo/athletic-pose/data/AthleticsPoseDataset'
PENN_ROOT = '/kaggle/input/datasets/kaushalbora18/penn-action-dataset/Penn_Action'

AP_GT_3D  = os.path.join(AP_ROOT, 'gt_markers3d_by_cam')       # .npz markers_h36m (T,17,3)
AP_GT_2D  = os.path.join(AP_ROOT, 'gt_markers2d_by_cam')       # .npy (T,17,3) x,y,conf
AP_DET_FT = os.path.join(AP_ROOT, 'det_markers2d_by_cam_ft')   # .npy detected keypoints

PENN_FRAMES = os.path.join(PENN_ROOT, 'frames')
PENN_LABELS = os.path.join(PENN_ROOT, 'labels')

# ============================================================
# DATASET CHECK
# ============================================================
print('Dataset check:')

paths = [
    ('AP gt_3d', AP_GT_3D),
    ('AP gt_2d', AP_GT_2D),
    ('AP det_ft', AP_DET_FT),
    ('Penn frames', PENN_FRAMES),
    ('Penn labels', PENN_LABELS),
]

for name, path in paths:
    ok = '✅' if os.path.exists(path) else '❌'
    count = len(os.listdir(path)) if os.path.exists(path) else 0
    print(f'{ok} {name}: {count} items')

# ============================================================
# ATHLETICS EVENTS
# ============================================================
events = sorted(os.listdir(AP_GT_3D))

print(f'\nAthletics events ({len(events)}):')

for ev in events:
    n = len(
        glob.glob(
            os.path.join(AP_GT_3D, ev, '**/*.npz'),
            recursive=True
        )
    )
    print(f'{ev:25s}: {n} clips')

Dataset check:
✅ AP gt_3d: 8 items
✅ AP gt_2d: 8 items
✅ AP det_ft: 8 items
✅ Penn frames: 2326 items
✅ Penn labels: 2326 items

Athletics events (8):
discus                   : 144 clips
hurdle                   : 688 clips
javelin                  : 112 clips
racewalk                 : 312 clips
running                  : 1272 clips
sd                       : 912 clips
shotput                  : 144 clips
sprint                   : 414 clips


In [10]:

# Cell 3 — Load Pretrained MotionBERT Checkpoint
import os
import torch

# Your uploaded pretrained checkpoint
MB_CKPT = '/kaggle/input/datasets/haruxo/motionbert/latest_epoch.bin'

# Verify it exists
assert os.path.exists(MB_CKPT), f'❌ Not found: {MB_CKPT}'
print(f'✅ Found pretrained checkpoint: {os.path.getsize(MB_CKPT)/1e6:.1f} MB')

# Load checkpoint
ckpt = torch.load(MB_CKPT, map_location='cpu')
print(f'Keys: {list(ckpt.keys())}')


✅ Found pretrained checkpoint: 170.0 MB
Keys: ['model_pos']


In [11]:
# Cell 4 — Build & Load MotionBERT Model
import torch.nn as nn
from lib.model.DSTformer import DSTformer

# Clean state dict (remove 'module.' prefix if present)
sd = ckpt.get('model_pos', ckpt.get('model', ckpt))
sd = {k.replace('module.', ''): v for k, v in sd.items()}

# Auto-detect model config from checkpoint
dim_feat = sd['blocks_st.0.attn_s.proj.weight'].shape[1]
mlp_hidden = sd['blocks_st.0.mlp_s.fc1.weight'].shape[0]
mlp_ratio = mlp_hidden // dim_feat
depth = sum(1 for k in sd if k.startswith('blocks_st.') and k.endswith('.norm1_s.weight'))
dim_in = sd['joints_embed.weight'].shape[1]  # Should be 3 (x,y,conf)

print(f'🔧 Auto-detected config:')
print(f'   dim_in  : {dim_in} (x,y,conf)')
print(f'   dim_feat: {dim_feat}')
print(f'   depth   : {depth}')
print(f'   mlp_ratio: {mlp_ratio}')

# Build MotionBERT model matching checkpoint
motionbert = DSTformer(
    dim_in=dim_in,      # 3 channels
    dim_out=3,          # 3D output
    dim_feat=dim_feat,
    dim_rep=dim_feat,
    depth=depth,
    num_heads=8,
    mlp_ratio=mlp_ratio,
    norm_layer=nn.LayerNorm,
    maxlen=WINDOW,      # 243 frames
    num_joints=17,
).to(device)

# Load pretrained weights
miss, unexp = motionbert.load_state_dict(sd, strict=False)
print(f'✅ Weights loaded: {len(miss)} missing, {len(unexp)} unexpected')

# Test forward pass
with torch.no_grad():
    dummy = torch.randn(1, WINDOW, 17, 3).to(device)
    out = motionbert(dummy)
    print(f'✅ Shape test: {dummy.shape} → {out.shape}')

print(f'🎉 MotionBERT ready for inference!')


🔧 Auto-detected config:
   dim_in  : 3 (x,y,conf)
   dim_feat: 512
   depth   : 5
   mlp_ratio: 2
✅ Weights loaded: 0 missing, 0 unexpected
✅ Shape test: torch.Size([1, 243, 17, 3]) → torch.Size([1, 243, 17, 3])
🎉 MotionBERT ready for inference!


In [12]:
# Cell 4b — Load Dataset Verification (OPTIONAL - for inference only, can SKIP)
import glob
import os
import numpy as np
import torch

def load_ap_clips_3ch(gt_3d_dir, det_2d_dir, gt_2d_dir):
    """Load AthleticsPose clips with 3-channel 2D keypoints (x,y,conf)"""
    files = sorted(glob.glob(os.path.join(gt_3d_dir, '**', '*.npz'), recursive=True))
    clips, errors = [], 0
    
    for f3d in files:
        try:
            rel = os.path.relpath(f3d, gt_3d_dir)
            stem = os.path.splitext(rel)[0]
            event = rel.split(os.sep)[0]
            
            # Load 3D ground truth
            npz = np.load(f3d)
            j3d = npz['markers_h36m'].astype(np.float32) / 1000.0  # mm → meters
            j3d -= j3d[:, 0:1, :]  # root-relative
            
            # Try detected first, then GT 2D
            for d2_dir in [det_2d_dir, gt_2d_dir]:
                f2d = os.path.join(d2_dir, stem + '.npy')
                if os.path.exists(f2d): 
                    break
            
            # Load 2D keypoints (T,17,3) or create zeros
            arr = (np.load(f2d).astype(np.float32) if os.path.exists(f2d) 
                   else np.zeros((j3d.shape[0], 17, 3), np.float32))
            
            j2d_3ch = arr[:, :, :3]  # Keep x,y,conf
            T = min(j2d_3ch.shape[0], j3d.shape[0])
            
            clips.append({
                'joints_2d': j2d_3ch[:T], 
                'joints_3d': j3d[:T], 
                'event': event, 
                'source': 'ap'
            })
        except Exception as e:
            errors += 1
    
    print(f'✅ AP (3ch): {len(clips)} clips ({errors} errors)')
    return clips

def load_penn_clips_3ch(frames_dir, labels_dir):
    """Load PennAction clips with synthetic 3-channel 2D keypoints"""
    import scipy.io as sio
    PENN_TO_COCO = {0:0, 1:5, 2:6, 3:7, 4:8, 5:9, 6:10, 7:11, 8:12, 9:13, 
                    10:14, 11:15, 12:16}
    files = sorted(glob.glob(os.path.join(labels_dir, '*.mat')))
    clips = []
    
    for lf in files:
        try:
            mat = sio.loadmat(lf)
            x, y = mat['x'].astype(np.float32), mat['y'].astype(np.float32)
            vis = mat['visibility'].astype(np.float32)
            T = x.shape[0]
            
            dims = mat.get('dimensions', np.array([[480, 640]]))
            H_i, W_i = float(dims[0, 0]), float(dims[0, 1] if dims.shape[1] > 1 else 640.0)
            
            # Create 3-channel keypoints (T,17,3): x_norm, y_norm, visibility
            j2d = np.zeros((T, 17, 3), np.float32)
            for pi, ci in PENN_TO_COCO.items():
                j2d[:, ci, 0] = (x[:, pi] / W_i) * 2 - 1.0  # x normalized
                j2d[:, ci, 1] = (y[:, pi] / H_i) * 2 - 1.0  # y normalized
                j2d[:, ci, 2] = vis[:, pi]  # confidence
            
            # Derived joints
            j2d[:, 0] = (j2d[:, 5] + j2d[:, 6]) / 2  # nose
            j2d[:, 8] = (j2d[:, 5] + j2d[:, 6]) / 2  # thorax
            j2d[:, 7] = (j2d[:, 11] + j2d[:, 12]) / 2  # spine
            
            clips.append({
                'joints_2d': j2d, 
                'joints_3d': None, 
                'action': str(mat['action'][0]), 
                'source': 'penn'
            })
        except:
            pass
    
    print(f'✅ Penn (3ch): {len(clips)} clips')
    return clips

def generate_pseudo_3d_3ch(penn_clips, model, window=243, device='cuda'):
    """Generate pseudo-3D for PennAction using MotionBERT"""
    model.eval()
    updated = []
    
    for clip in penn_clips:
        kp2 = torch.FloatTensor(clip['joints_2d'])  # (T,17,3)
        T = len(kp2)
        pred = np.zeros((T, 17, 3), np.float32)
        cnt = np.zeros(T, np.float32)
        
        with torch.no_grad():
            for s in range(0, T, window // 2):
                chunk = kp2[s:s + window]
                if len(chunk) < window:
                    chunk = torch.nn.functional.pad(
                        chunk.permute(2, 1, 0), (0, window - len(chunk)), 
                        mode='replicate').permute(2, 1, 0)
                
                inp = chunk.unsqueeze(0).to(device)  # (1,T,17,3)
                out = model(inp).squeeze(0).cpu().numpy()  # (T,17,3)
                
                ae = min(s + window, T)
                pred[s:ae] += out[:ae - s]
                cnt[s:ae] += 1
        
        c = clip.copy()
        c['joints_3d'] = pred / np.maximum(cnt[:, None, None], 1)
        updated.append(c)
    
    print(f'✅ Pseudo-3D (3ch) for {len(updated)} Penn clips')
    return updated

def align_scale(penn_clips, ap_clips):
    """Scale PennAction pseudo-3D to match AthleticsPose real-world scale"""
    ap_fem = np.mean([
        np.linalg.norm(c['joints_3d'][:, 11] - c['joints_3d'][:, 13], axis=-1).mean()
        for c in ap_clips
    ])
    penn_fem = np.mean([
        np.linalg.norm(c['joints_3d'][:, 11] - c['joints_3d'][:, 13], axis=-1).mean()
        for c in penn_clips
    ])
    scale = ap_fem / max(penn_fem, 1e-6)
    print(f'Scale: AP femur={ap_fem*1000:.1f}mm Penn={penn_fem*1000:.1f}mm factor={scale:.4f}')
    
    for c in penn_clips:
        c['joints_3d'] *= scale
    return penn_clips

# Only run if dataset paths exist (OPTIONAL for inference-only pipeline)
print("🔄 Checking dataset paths...")
if 'AP_GT_3D' in globals():
    ap_clips = load_ap_clips_3ch(AP_GT_3D, AP_DET_FT, AP_GT_2D)
    penn_clips = load_penn_clips_3ch(PENN_FRAMES, PENN_LABELS)
    penn_clips = generate_pseudo_3d_3ch(penn_clips, motionbert, WINDOW, device)
    penn_clips = align_scale(penn_clips, ap_clips)
    print(f'\n✅ Total: AP={len(ap_clips)} Penn={len(penn_clips)}')
    print(f'   Input shape: {ap_clips[0]["joints_2d"].shape} (T,17,3) ✅')
else:
    print("⏭️  Skipping dataset loading (inference-only mode)")


🔄 Checking dataset paths...
✅ AP (3ch): 3998 clips (0 errors)
✅ Penn (3ch): 2326 clips
✅ Pseudo-3D (3ch) for 2326 Penn clips
Scale: AP femur=101.2mm Penn=372.9mm factor=0.2714

✅ Total: AP=3998 Penn=2326
   Input shape: (420, 17, 3) (T,17,3) ✅


In [13]:
# Cell 5 — Inference Pipeline (YOLOv8n → MotionBERT)
import cv2
import numpy as np
import torch
from ultralytics import YOLO

# Set MotionBERT to eval mode
motionbert.eval()
print('✅ MotionBERT loaded from pretrained checkpoint (Cell 4)')

# Load YOLOv8n-pose (nano model - fastest)
yolo = YOLO('yolov8n-pose.pt')
print('✅ YOLOv8n-pose loaded (nano model)')

# COCO → H36M 17-joint mapping
COCO_TO_H36M = {1:12, 2:14, 3:16, 4:11, 5:13, 6:15, 10:0, 11:5, 12:7, 13:9, 14:6, 15:8, 16:10}

def yolo_to_h36m(coco_kp, scores_kp, W, H):
    """Convert YOLO COCO 17 keypoints → H36M 17 joints (x,y,conf normalized [-1,1])"""
    kp = np.zeros((17, 3), np.float32)
    norm = coco_kp.astype(np.float32).copy()
    
    # Normalize to [-1,1]
    norm[:, 0] = (norm[:, 0] / W) * 2 - 1.0
    norm[:, 1] = (norm[:, 1] / H) * 2 - 1.0
    
    # Map COCO joints to H36M
    for h36m_idx, coco_idx in COCO_TO_H36M.items():
        kp[h36m_idx, 0] = norm[coco_idx, 0]
        kp[h36m_idx, 1] = norm[coco_idx, 1]
        kp[h36m_idx, 2] = float(scores_kp[coco_idx]) if coco_idx < len(scores_kp) else 1.0
    
    # Derived joints (always confidence=1.0)
    kp[0, :2] = (norm[11, :2] + norm[12, :2]) / 2    # pelvis
    kp[0, 2] = 1.0
    kp[8, :2] = (norm[5, :2] + norm[6, :2]) / 2      # thorax  
    kp[8, 2] = 1.0
    kp[7, :2] = (kp[0, :2] + kp[8, :2]) / 2          # spine
    kp[7, 2] = 1.0
    kp[9, :2] = (kp[8, :2] + kp[10, :2]) / 2         # neck
    kp[9, 2] = 1.0
    
    return kp  # Shape: (17, 3) x,y,conf normalized

def run_inference(video_path, yolo_model, mb_model, target_fps=25, window=243, device='cuda', black_thresh=5.0):
    """Full pipeline: video → YOLO → MotionBERT → 3D poses"""
    cap = cv2.VideoCapture(video_path)
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H_vid = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    N = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, round(src_fps / target_fps))
    
    print(f'Video: {W}×{H_vid} {src_fps:.0f}fps {N} frames step={step}')
    
    kp2d_list, frames_rgb = [], []
    frame_idx, detected = 0, 0
    prev_kp = np.zeros((17, 3), np.float32)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
            
        if frame_idx % step == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # Skip black frames
            if rgb.mean() > black_thresh:
                res = yolo_model(frame, verbose=False)
                kp = prev_kp.copy()
                
                # Best person detection
                if (res and res[0].keypoints is not None and len(res[0].keypoints.xy) > 0):
                    scores = res[0].boxes.conf.cpu().numpy() if res[0].boxes is not None else np.array([0])
                    best = int(np.argmax(scores))
                    ckp = res[0].keypoints.xy[best].cpu().numpy()
                    kp_conf = res[0].keypoints.conf[best].cpu().numpy() if res[0].keypoints.conf is not None else np.ones(17, np.float32)
                    
                    if ckp.shape[0] == 17:
                        kp = yolo_to_h36m(ckp, kp_conf, W, H_vid)
                        prev_kp = kp.copy()
                        detected += 1
                
                kp2d_list.append(kp)
                frames_rgb.append(rgb)
        frame_idx += 1
    
    cap.release()
    
    # Stack keypoints: (T, 17, 3)
    kp2d = np.stack(kp2d_list)
    T = kp2d.shape[0]
    print(f'Extracted {T} frames | detection: {detected/max(T,1)*100:.1f}%')
    print(f'kp2d shape: {kp2d.shape} (T,17,3) ✅')
    
    # MotionBERT 3D lifting (sliding window)
    mb_model.eval()
    pred = np.zeros((T, 17, 3), np.float32)
    counts = np.zeros(T, np.float32)
    
    with torch.no_grad():
        for s in range(0, T, window // 2):
            chunk = kp2d[s:s + window]
            if len(chunk) < window:
                chunk = np.pad(chunk, ((0, window-len(chunk)), (0,0), (0,0)), mode='edge')
            
            inp = torch.FloatTensor(chunk).unsqueeze(0).to(device)  # (1,T,17,3)
            out = mb_model(inp).squeeze(0).cpu().numpy()           # (T,17,3)
            
            ae = min(s + window, T)
            pred[s:ae] += out[:ae - s]
            counts[s:ae] += 1
    
    poses_3d = pred / np.maximum(counts[:, None, None], 1)
    print(f'✅ poses_3d: {poses_3d.shape}')
    print(f'poses_3d range: {poses_3d.min():.3f}, {poses_3d.max():.3f}m')
    
    return kp2d, poses_3d, frames_rgb

# Your specific video path
VIDEO_PATH = '/kaggle/input/datasets/haruxo/test-video1/Discus_throw_technique_Outdoor_preparation_over_140_feet_41_meters_720P.mp4'
assert os.path.exists(VIDEO_PATH), f'Not found: {VIDEO_PATH}'

# Run inference!
kp2d, poses_3d, frames_rgb = run_inference(
    VIDEO_PATH, yolo, motionbert, 
    target_fps=25, window=WINDOW, device=device
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ MotionBERT loaded from pretrained checkpoint (Cell 4)
✅ YOLOv8n-pose loaded (nano model)
Video: 720×1280 60fps 640 frames step=2
Extracted 320 frames | detection: 100.0%
kp2d shape: (320, 17, 3) (T,17,3) ✅
✅ poses_3d: (320, 17, 3)
poses_3d range: -0.647, 0.706m


In [ ]:
import imageio
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cv2
import numpy as np
from IPython.display import Image as IPImage, display

# H36M bone connections (16 bones)
H36M_BONES = [
    (0,1),(1,2),(2,3),      # right leg
    (0,4),(4,5),(5,6),      # left leg  
    (0,7),(7,8),(8,9),(9,10),  # spine/upper body
    (8,11),(11,12),(12,13),    # left arm
    (8,14),(14,15),(15,16)     # right arm
]

# BGR colors for each bone (matching H36M_BONES order)
BONE_BGR = [
    (0,0,255), (0,0,255), (0,0,255),        # right leg - red
    (255,100,0), (255,100,0), (255,100,0),  # left leg - orange
    (0,220,0), (0,220,0), (0,220,0), (0,220,0),  # spine - green
    (0,165,255), (0,165,255), (0,165,255),  # left arm - cyan
    (255,0,220), (255,0,220), (255,0,220)   # right arm - magenta
]

def draw_overlay(frame_rgb, kp2d_frame):
    """Draw skeleton overlay on video frame using YOLO 2D keypoints"""
    H_f, W_f = frame_rgb.shape[:2]
    bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
    
    # Denormalize keypoints from [-1,1] to pixel coordinates [0,W],[0,H]
    px = np.zeros((17, 2), np.int32)
    px[:, 0] = ((kp2d_frame[:, 0] + 1) / 2 * W_f).clip(0, W_f-1).astype(np.int32)
    px[:, 1] = ((kp2d_frame[:, 1] + 1) / 2 * H_f).clip(0, H_f-1).astype(np.int32)
    
    # Detect visible joints (confidence > 0)
    det = np.any(kp2d_frame != 0, axis=1)
    
    # Draw bones
    for bi, (j1, j2) in enumerate(H36M_BONES):
        if det[j1] and det[j2]:
            cv2.line(bgr, tuple(px[j1]), tuple(px[j2]), 
                    BONE_BGR[bi], 3, cv2.LINE_AA)
    
    # Draw joints
    for j in range(17):
        if det[j]:
            # White filled circle
            cv2.circle(bgr, tuple(px[j]), 5, (255, 255, 255), -1, cv2.LINE_AA)
            # Dark outline
            cv2.circle(bgr, tuple(px[j]), 6, (40, 40, 40), 1, cv2.LINE_AA)
    
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

# Render overlay frames (max 300 for GIF)
T = min(len(frames_rgb), len(kp2d))
every_n = max(1, T // 300)
fids = list(range(0, T, every_n))

print(f'Rendering {len(fids)} overlay frames...')
overlay_frames = [draw_overlay(frames_rgb[fi], kp2d[fi]) for fi in fids]

# Save GIF
imageio.mimsave('/kaggle/working/overlay.gif', overlay_frames, fps=12)

# Save MP4
fH, fW = overlay_frames[0].shape[:2]
writer = cv2.VideoWriter('/kaggle/working/overlay.mp4', 
                        cv2.VideoWriter_fourcc(*'mp4v'), 12, (fW, fH))
for f in overlay_frames:
    writer.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
writer.release()

print('✅ overlay.gif + overlay.mp4 saved')
display(IPImage('/kaggle/working/overlay.gif'))


Rendering 320 overlay frames...
✅ overlay.gif + overlay.mp4 saved


In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import imageio
import cv2
import numpy as np
from IPython.display import Image as IPImage, display

# H36M bone connections (16 bones)
H36M_BONES = [
    (0,1),(1,2),(2,3),(0,4),(4,5),(5,6),     # legs
    (0,7),(7,8),(8,9),(9,10),                # spine/arms  
    (8,11),(11,12),(12,13),(8,14),(14,15),(15,16)  # arms
]

BONE_COLORS_3D = [
    '#e41a1c','#e41a1c','#e41a1c',      # right leg - red
    '#377eb8','#377eb8','#377eb8',      # left leg - blue  
    '#4daf4a','#4daf4a','#4daf4a','#4daf4a',  # spine - green
    '#ff7f00','#ff7f00','#ff7f00',      # left arm - orange
    '#984ea3','#984ea3','#984ea3'       # right arm - purple
]

# **FIX 1: Use ALL frames or dense sampling for smooth motion**
T = min(len(frames_rgb), len(poses_3d))
use_frames = min(T, 300)  # Max 300 frames for GIF
fids = list(range(0, use_frames, max(1, T//300)))  # DENSE sampling
print(f'Using {len(fids)}/{T} frames for smooth motion')

# **FIX 2: Consistent axis range**
all_c = np.stack([poses_3d[min(fi,T-1)] - poses_3d[min(fi,T-1), 0:1] for fi in fids])
g_r = max(float(np.abs(all_c).max()) * 1.3, 0.3)
print(f'Radius: {g_r:.3f}m')

# Create figure
plt.ioff()  # Non-interactive mode for speed
fig = plt.figure(figsize=(12, 5), facecolor='#111111')
ax_vid = fig.add_subplot(1, 2, 1)
ax_3d = fig.add_subplot(1, 2, 2, projection='3d')
rendered_3d = []

print('Rendering 3D animation...')
for i, fi in enumerate(fids):
    ax_vid.cla()
    ax_3d.cla()
    
    # Video frame
    ax_vid.imshow(frames_rgb[min(fi, len(frames_rgb)-1)])
    ax_vid.axis('off')
    ax_vid.set_title(f'Frame {fi}', color='white', fontsize=12)
    
    # 3D pose (pelvis-centered)
    pose_frame = min(fi, len(poses_3d)-1)
    kp3 = poses_3d[pose_frame].copy() - poses_3d[pose_frame, 0:1]
    
    # Bones
    for bi, (j1, j2) in enumerate(H36M_BONES):
        x_line = [kp3[j1, 0], kp3[j2, 0]]
        y_line = [kp3[j1, 2], kp3[j2, 2]] 
        z_line = [kp3[j1, 1], kp3[j2, 1]]
        ax_3d.plot(x_line, y_line, z_line, c=BONE_COLORS_3D[bi], 
                  linewidth=4, solid_capstyle='round')
    
    # Joints
    ax_3d.scatter(kp3[:, 0], kp3[:, 2], kp3[:, 1], c='white', 
                 s=60, zorder=10, edgecolors='black', linewidth=1, depthshade=False)
    
    # **FIX 3: Smooth rotation + better camera**
    angle = 45 + (i / max(len(fids)-1, 1)) * 360  # Full 360° rotation
    ax_3d.view_init(elev=10, azim=angle)
    
    # Style
    ax_3d.set_xlim(-g_r, g_r)
    ax_3d.set_ylim(-g_r, g_r)
    ax_3d.set_zlim(-g_r, g_r)
    ax_3d.set_facecolor('#111111')
    ax_3d.set_title('MotionBERT 3D Pose', color='white', fontsize=12, pad=20)
    
    # Clean axes
    for pane in [ax_3d.xaxis.pane, ax_3d.yaxis.pane, ax_3d.zaxis.pane]:
        pane.fill = False
        pane.set_edgecolor('none')
    for line in [ax_3d.xaxis.line, ax_3d.yaxis.line, ax_3d.zaxis.line]:
        line.set_color('none')
    ax_3d.grid(False)
    ax_3d.set_xticks([])
    ax_3d.set_yticks([])
    ax_3d.set_zticks([])
    
    # **FIX 4: Faster rendering**
    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    rendered_3d.append(buf.reshape(h, w, 4)[:, :, :3])
    
    if (i + 1) % 30 == 0:
        print(f'  {i+1}/{len(fids)}')

plt.close(fig)
plt.ion()  # Re-enable interactive mode

# Save outputs
print('Saving animations...')
imageio.mimsave('/kaggle/working/motionbert3d.gif', rendered_3d, fps=15)
fH, fW = rendered_3d[0].shape[:2]
writer = cv2.VideoWriter('/kaggle/working/motionbert3d.mp4', 
                        cv2.VideoWriter_fourcc(*'mp4v'), 15, (fW, fH))
for f in rendered_3d:
    writer.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
writer.release()

print('✅ motionbert3d.gif + motionbert3d.mp4 saved (SMOOTH MOTION!)')
display(IPImage('/kaggle/working/motionbert3d.gif'))
